In [1]:

import random
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.exceptions import ConvergenceWarning
import warnings
warnings.filterwarnings("ignore", category=ConvergenceWarning)
X, y = load_iris(return_X_y=True)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=0.25, random_state=42, stratify=y_train_val
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
POPULATION_SIZE = 10
GENERATIONS = 10
CROSSOVER_RATE = 0.8
MUTATION_RATE = 0.2
NEURONS = [4, 8, 16]
LEARNING_RATES = [0.001, 0.01, 0.1]
EPOCHS = [50, 100, 150]
random.seed(42)
np.random.seed(42)
def create_chromosome():
    return [
        random.randrange(len(NEURONS)),
        random.randrange(len(LEARNING_RATES)),
        random.randrange(len(EPOCHS))
    ]
population = [
    create_chromosome()
    for _ in range(POPULATION_SIZE)
]
def fitness(chromosome):
    neurons = NEURONS[chromosome[0]]
    learning_rate = LEARNING_RATES[chromosome[1]]
    epochs = EPOCHS[chromosome[2]]
    model = MLPClassifier(
        hidden_layer_sizes=(neurons,),
        learning_rate_init=learning_rate,
        max_iter=epochs,
        solver="adam",
        random_state=42
    )
    model.fit(X_train_scaled, y_train)
    predictions = model.predict(X_val_scaled)
    return accuracy_score(y_val, predictions)
def select_parent(population, scores):
    candidates = random.sample(
        range(len(population)), 3
    )
    winner = max(candidates, key=lambda i: scores[i])
    return population[winner][:]
def crossover(parent1, parent2):
    child1 = parent1[:]
    child2 = parent2[:]
    if random.random() < CROSSOVER_RATE:
        for i in range(len(parent1)):
            if random.random() < 0.5:
                child1[i], child2[i] = child2[i], child1[i]
    return child1, child2
def mutate(chromosome):
    limits = [
        len(NEURONS),
        len(LEARNING_RATES),
        len(EPOCHS)
    ]
    for i in range(len(chromosome)):
        if random.random() < MUTATION_RATE:
            chromosome[i] = random.randrange(limits[i])
    return chromosome
best_chromosome = None
best_fitness = -1.0
for generation in range(GENERATIONS):
    scores = [fitness(ind) for ind in population]
    generation_best = max(
        range(len(population)),
        key=lambda i: scores[i]
    )
    if scores[generation_best] > best_fitness:
        best_fitness = scores[generation_best]
        best_chromosome = population[generation_best][:]
    print(
        f"Generation {generation + 1}: "
        f"Best validation accuracy = "
        f"{scores[generation_best]:.4f}"
    )
    new_population = [best_chromosome[:]]
    while len(new_population) < POPULATION_SIZE:
        parent1 = select_parent(population, scores)
        parent2 = select_parent(population, scores)
        child1, child2 = crossover(parent1, parent2)
        child1 = mutate(child1)
        child2 = mutate(child2)
        new_population.append(child1)
        if len(new_population) < POPULATION_SIZE:
            new_population.append(child2)
    population = new_population
best_neurons = NEURONS[best_chromosome[0]]
best_lr = LEARNING_RATES[best_chromosome[1]]
best_epochs = EPOCHS[best_chromosome[2]]
print("\n--- BEST GA SOLUTION ---")
print("Hidden neurons:", best_neurons)
print("Learning rate:", best_lr)
print("Training epochs:", best_epochs)
print(f"Best validation accuracy: {best_fitness:.4f}")
final_scaler = StandardScaler()
X_train_val_scaled = final_scaler.fit_transform(X_train_val)
X_test_scaled = final_scaler.transform(X_test)
final_model = MLPClassifier(
    hidden_layer_sizes=(best_neurons,),
    learning_rate_init=best_lr,
    max_iter=best_epochs,
    solver="adam",
    random_state=42
)
final_model.fit(X_train_val_scaled, y_train_val)
test_predictions = final_model.predict(X_test_scaled)
test_accuracy = accuracy_score(y_test, test_predictions)
print(f"Final test accuracy: {test_accuracy:.4f}")
print("Test accuracy (%):", round(test_accuracy * 100, 2))

Generation 1: Best validation accuracy = 0.9333
Generation 2: Best validation accuracy = 0.9333
Generation 3: Best validation accuracy = 0.9333
Generation 4: Best validation accuracy = 0.9333
Generation 5: Best validation accuracy = 0.9333
Generation 6: Best validation accuracy = 0.9333
Generation 7: Best validation accuracy = 0.9333
Generation 8: Best validation accuracy = 0.9333
Generation 9: Best validation accuracy = 0.9333
Generation 10: Best validation accuracy = 0.9333

--- BEST GA SOLUTION ---
Hidden neurons: 16
Learning rate: 0.01
Training epochs: 50
Best validation accuracy: 0.9333
Final test accuracy: 0.9333
Test accuracy (%): 93.33
